### Import Libraries

In [1]:
# Standard library
import random
from pathlib import Path

# Third-party libraries
import numpy as np
import pandas as pd

# NLP
from nltk.corpus import stopwords

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix, classification_report

### Config

In [2]:
# Constants
RANDOM_STATE = 42
TARGET_COL = "label"
TEXT_COL = "text"
TEST_SIZE = 0.25

# Base path
BASE_DIR = Path.cwd().resolve().parents[1]

# Paths
DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DATA_PATH = PROCESSED_DIR / "cleaned_sms.csv"

np.random.seed(RANDOM_STATE) # ensures reproducibility for NumPy-based operations
random.seed(RANDOM_STATE) # ensures reproducibility for Python's random module

### Load Dataset

In [3]:
df = pd.read_csv(PROCESSED_DATA_PATH)
print("Shape:", df.shape)
print("\nData Types:\n", df.dtypes)

Shape: (5157, 3)

Data Types:
 label       object
raw_text    object
text        object
dtype: object


In [4]:
# Preview
df.head(5)

,label,raw_text,text
0,ham,"Go until jurong point, crazy.. Available only ...","Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...,Ok lar... Joking wif you oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...,YOU dun say so early hor... YOU c already then...
4,ham,"Nah I don't think he goes to usf, he lives aro...","Nah I do not think he goes to usf, he lives ar..."


In [5]:
# Define Features and Target
X = df[TEXT_COL]
y = df[TARGET_COL].map({'ham': 0, 'spam': 1})

In [6]:
# Class distribution
print(y.value_counts(normalize=True).rename("ratio"))

label
0    0.875703
1    0.124297
Name: ratio, dtype: float64


**Observation**
- Processed dataset loaded successfully with 5157 rows and 3 columns
- `text` column will be used as input features, and `label` as target
- Class distribution remains imbalanced (~87.6% ham), consistent with preprocessing stage

### Train/Test split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=TEST_SIZE, 
                                                    random_state=RANDOM_STATE,
                                                    stratify=y)

In [8]:
# Shape check
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (3867,)
X_test : (1290,)
y_train: (3867,)
y_test : (1290,)


In [9]:
# Distribution check
print("\nTrain distribution:")
print(y_train.value_counts(normalize=True).rename("ratio"))

print("\nTest distribution:")
print(y_test.value_counts(normalize=True).rename("ratio"))


Train distribution:
label
0    0.875614
1    0.124386
Name: ratio, dtype: float64

Test distribution:
label
0    0.875969
1    0.124031
Name: ratio, dtype: float64


**Observation**
- Dataset successfully split into train and test sets (75/25 split)
- Train set contains 3867 samples and test set contains 1290 samples
- Class distribution is preserved across train and test sets (≈87.6% ham, ≈12.4% spam), confirming correct stratification
- This ensures balanced representation of classes and supports reliable model evaluation

### Bag of Words Vectorization (CountVectorizer)

In [10]:
# Custom stopwords (retain negation words for sentiment)
stop_words = list(set(stopwords.words('english')) - {'not', 'no', 'never'})

# Initialize vectorizer
bow_vectorizer = CountVectorizer(
    lowercase=True,
    stop_words=stop_words,
    min_df=2   
)

# Fit on train and transform
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

# Vocabulary
features = bow_vectorizer.get_feature_names_out()

In [11]:
# Overview
sparsity = 1 - (X_train_bow.nnz / (X_train_bow.shape[0] * X_train_bow.shape[1]))

print("No. of features:", len(features))
print("Sample features:", features[:10])
print("Train shape:", X_train_bow.shape)
print("Test shape:", X_test_bow.shape)
print(f"Sparsity: {sparsity:.4f}")

No. of features: 2950
Sample features: ['00' '000' '008704050406' '02' '0207' '03' '04' '05' '050703' '0578']
Train shape: (3867, 2950)
Test shape: (1290, 2950)
Sparsity: 0.9976


**Observation**
- Bag of Words created a feature space of 2950 tokens after removing rare words (min_df=2), reducing noise and improving generalization
- Vocabulary is built only from training data, avoiding data leakage
- Negation words ('not', 'no', 'never') are retained to preserve sentiment and semantic meaning
- Representation is highly sparse (~99.76% zeros), which is typical for text data
- Sparse high-dimensional features are well-suited for linear models like Logistic Regression
- BoW captures word frequency but ignores context and word order

### Train Logistic Regression Model

In [12]:
# Check class imbalance
print(y_test.value_counts(normalize=True))

label
0    0.875969
1    0.124031
Name: proportion, dtype: float64


#### 1. Baseline Logistic Regression

In [13]:
# Using 'liblinear' solver as it works well for small datasets and sparse data (like BoW)
lr_bow_baseline = LogisticRegression(max_iter=1000, solver="liblinear")
lr_bow_baseline.fit(X_train_bow, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'liblinear'
,max_iter,1000
,multi_class,'deprecated'


#### Baseline LR Evaluation on Test Set

In [14]:
# Probabilities for ROC-AUC
lr_bow_baseline_proba = lr_bow_baseline.predict_proba(X_test_bow)[:, 1]
lr_bow_baseline_auc = roc_auc_score(y_test, lr_bow_baseline_proba)

# Predictions for F1
lr_bow_baseline_pred = lr_bow_baseline.predict(X_test_bow)
lr_bow_baseline_f1 = f1_score(y_test, lr_bow_baseline_pred)

print(f"Baseline Logistic Regression Test ROC-AUC: {lr_bow_baseline_auc:.4f}")
print(f"Baseline Logistic Regression Test F1 Score: {lr_bow_baseline_f1:.4f}")

Baseline Logistic Regression Test ROC-AUC: 0.9876
Baseline Logistic Regression Test F1 Score: 0.9000


Observation:
- Dataset is imbalanced (~12% spam), so ROC-AUC is used as the primary metric to evaluate ranking performance
- F1 score (~0.90) complements ROC-AUC by capturing the balance between precision and recall for the minority (spam) class
- Logistic Regression achieves strong performance (ROC-AUC ~0.988), indicating high separability between spam and ham messages
- BoW features provide sufficient signal for effective classification even without hyperparameter tuning
- High performance with a simple linear model suggests the dataset is largely linearly separable

#### 2. Tuning Logistic Regression 
- Hyperparameters
    - Regularization strength (C)

In [15]:
# Cross-validation strategy
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Parameter grid
param_grid = {
    "C": [0.01, 0.1, 1, 10, 100]
}

# Grid search
grid = GridSearchCV(
    estimator=LogisticRegression(
        max_iter=1000,
        solver="liblinear" # Using 'liblinear' solver as it works well for small datasets and sparse data (like BoW)
        ),
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1
)

grid.fit(X_train_bow, y_train)

,estimator,LogisticRegre...r='liblinear')
,param_grid,"{'C': [0.01, 0.1, ...]}"
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l2'


In [16]:
cv_results = pd.DataFrame(grid.cv_results_)
display(cv_results[["param_C", "mean_test_score"]])

,param_C,mean_test_score
0,0.01,0.970270
1,0.10,0.981188
2,1.00,0.983845
3,10.00,0.982309
4,100.00,0.979256


**Observation**
- ROC-AUC improves with increasing C up to ~1, after which performance plateaus
- Indicates that reducing regularization initially helps the model fit better
- Beyond C=1, additional flexibility does not improve performance, suggesting diminishing returns
- Stable performance across C values indicates low variance and a strong linear signal in the data

In [17]:
lr_bow_tuned = grid.best_estimator_
lr_bow_tuned_cv_auc = grid.best_score_

print(f"Best C: {grid.best_params_['C']}")
print(f"Tuned Logistic Regression CV ROC-AUC: {lr_bow_tuned_cv_auc:.4f}")

Best C: 1
Tuned Logistic Regression CV ROC-AUC: 0.9838


#### Tuned LR Evaluation on Test Set

In [18]:
# Probabilities for ROC-AUC
lr_bow_tuned_proba = lr_bow_tuned.predict_proba(X_test_bow)[:, 1]
lr_bow_tuned_auc = roc_auc_score(y_test, lr_bow_tuned_proba)

# Predictions for F1
lr_bow_tuned_pred = lr_bow_tuned.predict(X_test_bow)
lr_bow_tuned_f1 = f1_score(y_test, lr_bow_tuned_pred)

print(f"Tuned Logistic Regression Test ROC-AUC: {lr_bow_tuned_auc:.4f}")
print(f"Tuned Logistic Regression Test F1 Score: {lr_bow_tuned_f1:.4f}")

Tuned Logistic Regression Test ROC-AUC: 0.9876
Tuned Logistic Regression Test F1 Score: 0.9000


### Baseline vs Regularization LR

In [19]:
print(f"Baseline Logistic Regression Test ROC-AUC   : {lr_bow_baseline_auc:.4f}")
print(f"Tuned Logistic Regression Test ROC-AUC      : {lr_bow_tuned_auc:.4f}")

print(f"Baseline Logistic Regression Test F1 Score  : {lr_bow_baseline_f1:.4f}")
print(f"Tuned Logistic Regression Test F1 Score     : {lr_bow_tuned_f1:.4f}")

Baseline Logistic Regression Test ROC-AUC   : 0.9876
Tuned Logistic Regression Test ROC-AUC      : 0.9876
Baseline Logistic Regression Test F1 Score  : 0.9000
Tuned Logistic Regression Test F1 Score     : 0.9000


**Observation:**
- Baseline and tuned Logistic Regression achieve identical test performance (ROC-AUC ~0.988, F1 ~0.90)
- Despite slight improvements during cross-validation, tuning does not translate to better generalization on the test set
- Indicates the baseline model is already near optimal for this feature representation
- Performance is driven primarily by the BoW features rather than model hyperparameters
- Confirms that the dataset is highly linearly separable with low variance

### Tf-Idf Vectorization (TfidfVectorizer)

In [20]:
# Custom stopwords (retain negation words for sentiment)
stop_words = list(set(stopwords.words('english')) - {'not', 'no', 'never'})

# Initialize vectorizer
tf_idf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words=stop_words,
    min_df=2,
    sublinear_tf=True
)

# Fit on train and transform
X_train_tf_idf = tf_idf_vectorizer.fit_transform(X_train)
X_test_tf_idf = tf_idf_vectorizer.transform(X_test)

# Vocabulary
features = tf_idf_vectorizer.get_feature_names_out()

In [21]:
# Overview
sparsity = 1 - (X_train_tf_idf.nnz / (X_train_tf_idf.shape[0] * X_train_tf_idf.shape[1]))

print("No. of features:", len(features))
print("Sample features:", features[:10])
print("Train shape:", X_train_tf_idf.shape)
print("Test shape:", X_test_tf_idf.shape)
print(f"Sparsity: {sparsity:.4f}")

No. of features: 2950
Sample features: ['00' '000' '008704050406' '02' '0207' '03' '04' '05' '050703' '0578']
Train shape: (3867, 2950)
Test shape: (1290, 2950)
Sparsity: 0.9976


**Observation:**
- Tf-Idf created a feature space of 2950 tokens after removing rare words (min_df=2), reducing noise and improving generalization
- Vocabulary is built only from training data, avoiding data leakage
- Negation words ('not', 'no', 'never') are retained to preserve sentiment and semantic meaning
- Representation is highly sparse (~99.76% zeros), which is typical for text data
- Sparse high-dimensional features are well-suited for linear models like Logistic Regression
- Tf-Idf captures word frequency but ignores context and word order

### Train Logistic Regression Model

In [22]:
# Check class imbalance
print(y_test.value_counts(normalize=True))

label
0    0.875969
1    0.124031
Name: proportion, dtype: float64


#### 1. Baseline Logistic Regression

In [23]:
# Using 'liblinear' solver as it works well for small datasets and sparse data (like Tf-Idf)
lr_tf_idf_baseline = LogisticRegression(max_iter=1000, solver="liblinear")
lr_tf_idf_baseline.fit(X_train_tf_idf, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'liblinear'
,max_iter,1000
,multi_class,'deprecated'


#### Baseline LR Evaluation on Test Set

In [24]:
# Probabilities for ROC-AUC
lr_tf_idf_baseline_proba = lr_tf_idf_baseline.predict_proba(X_test_tf_idf)[:, 1]
lr_tf_idf_baseline_auc = roc_auc_score(y_test, lr_tf_idf_baseline_proba)

# Predictions for F1
lr_tf_idf_baseline_pred = lr_tf_idf_baseline.predict(X_test_tf_idf)
lr_tf_idf_baseline_f1 = f1_score(y_test, lr_tf_idf_baseline_pred)

print(f"Baseline Logistic Regression Test ROC-AUC: {lr_tf_idf_baseline_auc:.4f}")
print(f"Baseline Logistic Regression Test F1 Score: {lr_tf_idf_baseline_f1:.4f}")

Baseline Logistic Regression Test ROC-AUC: 0.9885
Baseline Logistic Regression Test F1 Score: 0.8014


**Observation:**
- Dataset is imbalanced (~12% spam), so ROC-AUC is used as the primary evaluation metric
- TF-IDF achieves slightly higher ROC-AUC (~0.988), indicating strong ranking performance
- However, F1 score (~0.80) is lower than BoW, suggesting weaker classification performance at the default threshold
- TF-IDF downweights frequently occurring words, which may reduce the impact of important spam-related keywords
- Compared to BoW, TF-IDF provides better normalization but may lose some discriminative power in this dataset
- Overall performance indicates that feature representation significantly impacts model behavior and metric outcomes

### BoW vs TF-IDF Comparison

In [25]:
# Comparison table
comparison_df = pd.DataFrame({
    "Representation": ["Bag of Words", "TF-IDF"],
    "ROC-AUC": [lr_bow_baseline_auc, lr_tf_idf_baseline_auc],
    "F1 Score": [lr_bow_baseline_f1, lr_tf_idf_baseline_f1]
})

print(comparison_df.to_string(index=False))

Representation  ROC-AUC  F1 Score
  Bag of Words 0.987561  0.900000
        TF-IDF 0.988462  0.801444


**Observation: Model Performance Comparison**
- Both BoW and TF-IDF achieve very high ROC-AUC (~0.988), indicating strong ranking performance
- TF-IDF slightly outperforms BoW in ROC-AUC (0.9885 vs 0.9876)
- However, BoW significantly outperforms TF-IDF in F1 score (0.90 vs 0.80)
- This indicates that while TF-IDF ranks predictions slightly better, BoW produces more accurate class predictions at the default threshold

**Observation: Metric-Level Insight**
- ROC-AUC evaluates ranking performance across thresholds, whereas F1 depends on a fixed threshold (0.5)
- TF-IDF improves ranking ability but leads to poorer classification balance (precision vs recall)
- BoW provides a stronger decision boundary for this dataset, leading to higher F1 score
- This highlights that different metrics capture different aspects of model performance

**Observation: Feature Representation Insight**
- BoW uses raw word counts, preserving the importance of frequently occurring spam-related keywords
- TF-IDF downweights commonly occurring words, which can reduce the impact of strong spam indicators
- In this dataset, frequency of keywords (e.g., "free", "win") is highly discriminative
- As a result, BoW retains stronger signals for classification compared to TF-IDF

**Observation: Practical Interpretation**
- Both representations produce similar feature spaces (2950 tokens, ~99.76% sparsity)
- Performance differences arise from weighting schemes rather than feature availability
- TF-IDF normalization is beneficial for general text tasks but may reduce discriminative power in keyword-driven problems like spam detection

### Final Decision

- Bag of Words is preferred for this dataset due to its superior F1 score and strong classification performance
- TF-IDF does not provide meaningful improvement and may reduce classification effectiveness
- Further improvements should focus on feature engineering (e.g., n-grams) rather than changing the vectorization method

### Motivation for n-grams
- Current representations (BoW, TF-IDF) use **unigrams (single words)**, which ignore word order and local context  
- However, in many NLP tasks, meaning is often conveyed through **phrases rather than individual words**  

- For example:
  - "free" vs **"free entry"**
  - "call" vs **"call now"**

- Unigrams treat these words independently, potentially losing important semantic signals  
- n-grams (e.g., bigrams) capture **adjacent word combinations**, allowing the model to learn more meaningful patterns  

- In spam detection, phrases are often stronger indicators than individual words  
- Therefore, incorporating n-grams can improve the model’s ability to detect contextual patterns and enhance classification performance

### Bag of Words Vectorization with n-grams (CountVectorizer)

In [26]:
# Custom stopwords (retain negation words for sentiment)
stop_words = list(set(stopwords.words('english')) - {'not', 'no', 'never'})

# Initialize vectorizer
n_bow_vectorizer = CountVectorizer(
    lowercase=True,
    stop_words=stop_words,
    min_df=2,
    ngram_range=(1,2) 
)

# Fit on train and transform
X_train_bow_ngram = n_bow_vectorizer.fit_transform(X_train)
X_test_bow_ngram = n_bow_vectorizer.transform(X_test)

# Vocabulary
n_bow_features = n_bow_vectorizer.get_feature_names_out()

In [27]:
# Overview
sparsity = 1 - (X_train_bow_ngram.nnz / (X_train_bow_ngram.shape[0] * X_train_bow_ngram.shape[1]))

print("No. of features:", len(n_bow_features))
print("Sample features:", n_bow_features[:10])
print("Train shape:", X_train_bow_ngram.shape)
print("Test shape:", X_test_bow_ngram.shape)
print(f"Sparsity: {sparsity:.4f}")

No. of features: 5705
Sample features: ['00' '00 sub' '000' '000 bonus' '000 cash' '000 prize' '000 xmas'
 '008704050406' '008704050406 sp' '02']
Train shape: (3867, 5705)
Test shape: (1290, 5705)
Sparsity: 0.9984


**Observation**
- Feature space increased from 2950 to 5705 tokens after introducing bigrams, capturing phrase-level patterns
- Representation became more sparse (~99.84% zeros) due to higher dimensionality
- Bigrams (e.g., "free entry", "call now") introduce contextual information beyond individual words

### Train Logistic Regression Model (n-gram features)

In [28]:
# Check class imbalance
print(y_test.value_counts(normalize=True))

label
0    0.875969
1    0.124031
Name: proportion, dtype: float64


#### 1. Baseline Logistic Regression

In [29]:
# Using 'liblinear' solver as it works well for small datasets and sparse data (like BoW)
lr_n_bow_baseline = LogisticRegression(max_iter=1000, solver="liblinear")
lr_n_bow_baseline.fit(X_train_bow_ngram, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'liblinear'
,max_iter,1000
,multi_class,'deprecated'


#### Baseline LR Evaluation on Test Set

In [30]:
# Probabilities for ROC-AUC
lr_n_bow_baseline_proba = lr_n_bow_baseline.predict_proba(X_test_bow_ngram)[:, 1]
lr_n_bow_baseline_auc = roc_auc_score(y_test, lr_n_bow_baseline_proba)

# Predictions for F1
lr_n_bow_baseline_pred = lr_n_bow_baseline.predict(X_test_bow_ngram)
lr_n_bow_baseline_f1 = f1_score(y_test, lr_n_bow_baseline_pred)

print(f"Baseline Logistic Regression Test (n-gram features) ROC-AUC: {lr_n_bow_baseline_auc:.4f}")
print(f"Baseline Logistic Regression Test (n-gram features) F1 Score: {lr_n_bow_baseline_f1:.4f}")

Baseline Logistic Regression Test (n-gram features) ROC-AUC: 0.9870
Baseline Logistic Regression Test (n-gram features) F1 Score: 0.8986


### Comparison: Unigram vs n-gram

In [31]:
comparison_df = pd.DataFrame({
    "Representation": ["Unigram BoW", "n-gram BoW"],
    "ROC-AUC": [lr_bow_baseline_auc, lr_n_bow_baseline_auc],
    "F1 Score": [lr_bow_baseline_f1, lr_n_bow_baseline_f1]
})

print(comparison_df.to_string(index=False))

Representation  ROC-AUC  F1 Score
   Unigram BoW 0.987561  0.900000
    n-gram BoW 0.986991  0.898649


**Observations:**
- Unigram BoW slightly outperforms n-gram BoW on both ROC-AUC (0.9876 vs 0.9870) and F1 score (0.9000 vs 0.8986)
- While n-grams capture local context (e.g., "free entry", "call now"), they do not provide additional discriminative power for this dataset
- The slight performance drop suggests that added n-gram features introduce noise and dilute strong unigram signals
- Results indicate that spam classification in this dataset is primarily driven by individual keyword presence rather than phrase-level context

### Final Insight

- Increasing feature complexity does not guarantee performance improvement  
- In this dataset, simple unigram features are sufficient to capture the underlying signal  
- This highlights the importance of validating feature engineering decisions empirically rather than assuming more complex representations will perform better

### Motivation for Error Analysis

- Evaluation metrics (ROC-AUC, F1 score) provide an overall measure of model performance but do not explain where or why the model makes mistakes  
- Understanding misclassifications helps identify specific weaknesses in the model and feature representation  

- For example:
  - Spam messages missed by the model (false negatives)
  - Legitimate messages incorrectly flagged as spam (false positives)

- Analyzing such errors reveals patterns that are not visible through aggregate metrics  
- This enables targeted improvements, such as:
  - refining features
  - adjusting decision thresholds
  - improving data representation  

- Therefore, error analysis is a critical step to move from model evaluation to model improvement

### Identify Misclassified Samples

In [32]:
# Getting y_pred using best model till now
y_pred = lr_bow_baseline.predict(X_test_bow)

# Misclassified Samples
misclassified_mask = (y_test != y_pred)
misclassified_df = pd.DataFrame({
    "text": X_test,
    "true label": y_test,
    "predicted label": y_pred
}).loc[misclassified_mask]

print(f"Total samples {X_test.shape[0]}")
print(f"Misclassified Samples {misclassified_df.shape[0]}")
print(f"Error Rate: {misclassified_df.shape[0]/X_test.shape[0]:.2%}")


Total samples 1290
Misclassified Samples 30
Error Rate: 2.33%


**Observations:**
- Only 30 out of 1290 test samples are misclassified (~2.3% error rate), indicating strong overall performance  
- Despite high performance, error analysis helps uncover specific weaknesses not visible through aggregate metrics  

### Inspect Misclassified Samples

#### 1. False Negatives (Actual = 1, Predicted = 0)

In [33]:
print("False Negatives (Spam predicted as Ham):\n")

fn_df = misclassified_df[
    (misclassified_df["true label"] == 1) &
    (misclassified_df["predicted label"] == 0)
]
print(f"False Negatives: {len(fn_df)}\n")

for _, row in fn_df.sample(min(4, len(fn_df)), random_state=RANDOM_STATE).iterrows():
    print("TEXT :", row["text"])
    print("TRUE LABEL :", row["true label"])
    print("PREDICTED LABEL :", row["predicted label"])
    print("-" * 60)

False Negatives (Spam predicted as Ham):

False Negatives: 25

TEXT : 1000's of girls many local 2 you who r virgins 2 this & r ready 2 4fil you are every sexual need. Can you 4fil theirs? text CUTE to 69911(£1.50p. m)
TRUE LABEL : 1
PREDICTED LABEL : 0
------------------------------------------------------------
TEXT : Monthly password for wap. mobsi.com is 391784. Use your wap phone not PC.
TRUE LABEL : 1
PREDICTED LABEL : 0
------------------------------------------------------------
TEXT : For sale - arsenal dartboard. Good condition but no doubles or trebles!
TRUE LABEL : 1
PREDICTED LABEL : 0
------------------------------------------------------------
TEXT : FreeMsg>FAV XMAS TONES!Reply REAL
TRUE LABEL : 1
PREDICTED LABEL : 0
------------------------------------------------------------


#### 2. False Positives (Actual = 0, Predicted = 1)

In [34]:
print("False Positives (Ham predicted as Spam):\n")

fp_df = misclassified_df[
    (misclassified_df["true label"] == 0) &
    (misclassified_df["predicted label"] == 1)
]
print(f"False Positives: {len(fp_df)}\n")

for _, row in fp_df.sample(min(4, len(fp_df)), random_state=RANDOM_STATE).iterrows():
    print("TEXT :", row["text"])
    print("TRUE LABEL :", row["true label"])
    print("PREDICTED LABEL :", row["predicted label"])
    print("-" * 60)

False Positives (Ham predicted as Spam):

False Positives: 5

TEXT : Plz note: if anyone calling from a mobile Co. & asks you to type # <#> or # <#> . Do not do so. Disconnect the call,coz it iz an attempt of 'terrorist' to make use of the sim card no. Itz confirmd by nokia n motorola n has been verified by CNN IBN.
TRUE LABEL : 0
PREDICTED LABEL : 1
------------------------------------------------------------
TEXT : Please protect yourself from e-threats. SIB never asks for sensitive information like Passwords,ATM/SMS PIN thru email. Never share your password with anybody.
TRUE LABEL : 0
PREDICTED LABEL : 1
------------------------------------------------------------
TEXT : Sir, i am waiting for your call, once free please call me.
TRUE LABEL : 0
PREDICTED LABEL : 1
------------------------------------------------------------
TEXT : Was actually sleeping and still might when you call back. So a text is gr8. You rock sis. Will send you a text wen i wake.
TRUE LABEL : 0
PREDICTED LABEL 

**Observations:**
- False Negatives (25) are significantly higher than False Positives (5), indicating the model tends to label messages as `ham` unless strong spam signals are present
- Many missed spam messages lack strong spam-indicative keywords (e.g., "free", "win", "call", "now"), making them harder for BoW to capture  
Some false negatives resemble normal conversational or informational text, making them indistinguishable based on keyword frequency alone
- False Positives often contain warning or alert-like language (e.g., "protect", "password", "urgent"), which overlaps with spam-like vocabulary  
- BoW relies on keyword presence without context, leading to confusion when legitimate messages share vocabulary with `spam`
- Overall, errors highlight the limitation of BoW in capturing semantic meaning and context beyond individual words  

### Confusion Matrix

In [35]:
cm = confusion_matrix(y_test, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual: ham (0)", "Actual: spam (1)"],
    columns=["Pred: ham (0)", "Pred: spam (1)"]
)
display(cm_df)

,Pred: ham (0),Pred: spam (1)
Actual: ham (0),1125,5
Actual: spam (1),25,135


**Observations:**
- The model correctly classifies most `ham` messages (1125/1130), resulting in very few false positives  
- However, it misses a portion of `spam` messages (25 cases), indicating a recall limitation for the `spam` class  
- Indicates the model is slightly conservative in predicting `spam`, prioritizing fewer false positives at the cost of missing some `spam` messages

### Classification Report

In [36]:
report = classification_report(
    y_test,
    y_pred,
    target_names=["ham (0)", "spam (1)"],
    output_dict=True
)

report_df = pd.DataFrame(report).transpose()
display(report_df)

,precision,recall,f1-score,support
ham (0),0.978261,0.995575,0.986842,1130.000000
spam (1),0.964286,0.843750,0.900000,160.000000
accuracy,0.976744,0.976744,0.976744,0.976744
macro avg,0.971273,0.919663,0.943421,1290.000000
weighted avg,0.976528,0.976744,0.976071,1290.000000


**Observations:**
- Overall accuracy is high (~97.7%), but influenced by class imbalance  
- `ham` class shows near-perfect performance (high precision and recall)  
- `spam` precision is high (~0.964), but recall is lower (~0.844), confirming that some spam messages are missed  
- F1-score (~0.90) reflects this precision–recall tradeoff  
- Improving spam recall (e.g., via threshold tuning or better features) could further enhance performance  

### Key Insights

- The model relies heavily on the presence of strong spam-related keywords for classification  
- Spam messages lacking explicit keywords are often misclassified as `ham` (false negatives)  
- The model demonstrates high precision but comparatively lower recall for the `spam` class  
- Overlap in vocabulary between spam and legitimate alert messages leads to occasional false positives  
- Overall, the model performs well but struggles with subtle or context-dependent `spam` patterns  

### Summary of Vectorization Techniques and Key Learnings

#### 1. Approaches Evaluated

The following text vectorization techniques were implemented and evaluated:

- **Bag of Words (BoW - Unigram)**  
  - Converts text into frequency-based features  
  - Simple, interpretable, and effective baseline  

- **TF-IDF (Term Frequency–Inverse Document Frequency)**  
  - Adjusts word importance by down-weighting frequent terms  
  - Aims to improve generalization by reducing noise  

- **n-grams (BoW with bigrams)**  
  - Extends BoW by including adjacent word combinations  
  - Captures limited local context (e.g., "free entry", "call now")  

#### 2. Performance Comparison

In [37]:
comparison_df = pd.DataFrame({
    "Representation": ["BoW (Unigram)", "TF-IDF", "BoW (n-gram)"],
    "ROC-AUC": [
        lr_bow_baseline_auc,
        lr_tf_idf_baseline_auc,
        lr_n_bow_baseline_auc
    ],
    "F1 Score": [
        lr_bow_baseline_f1,
        lr_tf_idf_baseline_f1,
        lr_n_bow_baseline_f1
    ]
})

# Round for readability
comparison_df[["ROC-AUC", "F1 Score"]] = comparison_df[["ROC-AUC", "F1 Score"]].round(4)

display(comparison_df)

,Representation,ROC-AUC,F1 Score
0,BoW (Unigram),0.9876,0.9000
1,TF-IDF,0.9885,0.8014
2,BoW (n-gram),0.9870,0.8986


**Observations:**
- All models achieve strong ROC-AUC (~0.98), indicating good ranking ability  
- **BoW (Unigram)** performs best in terms of F1 score (~0.90), making it the most effective for classification  
- **TF-IDF** slightly improves ranking performance (ROC-AUC), but shows lower F1 at the default threshold which suggests the model is well-ranked but not optimally thresholded for classification 
- **n-grams** increased feature space but did not provide meaningful improvement for this dataset, suggesting that unigram features already capture most of the signal
- **F1 score** depends on the classification threshold and reflects the balance between precision and recall for the minority class

#### 3. Key Learnings

- **Feature representation has a greater impact than model tuning**  
  - Logistic Regression performed consistently well across experiments  
  - Improvements (or drops) were driven primarily by feature changes  

- **Simpler models can perform extremely well**  
  - Unigram BoW captured sufficient signal for this dataset  

- **More features do not guarantee better performance**  
  - n-grams increased dimensionality but introduced noise  

- **Different metrics capture different behaviors**  
  - ROC-AUC measures ranking ability  
  - F1 reflects actual classification performance at a threshold  
  - TF-IDF improved ranking performance but resulted in suboptimal classification at the default threshold  

- **Model performance depends heavily on dataset characteristics**  
  - Spam detection here is largely driven by keyword presence  

#### 4. Final Model Selection

- **Selected Model:** Bag of Words (Unigram) + Logistic Regression  

**Reasoning:**
- Selected based on evaluation metric aligned with task (F1 for spam detection)
- Achieves the best balance between precision and recall (highest F1 score)  
- Simple, efficient, and interpretable  
- No additional complexity from feature engineering or tuning required  

#### 5. Limitations

- BoW ignores word order and deeper semantic context  
- Model relies heavily on keyword presence  
- Struggles with subtle or implicit spam messages  
- Limited ability to generalize beyond seen vocabulary patterns  

#### 6. Future Improvements

- Tune classification threshold to improve recall/precision trade-off 
- Explore more advanced feature representations:
  - TF-IDF with threshold tuning  
  - Word embeddings (Word2Vec, GloVe)  
  - Contextual embeddings (BERT)  
- Incorporate domain-specific feature engineering  
- Use larger or more diverse datasets to improve generalization

### Final Insight

- Increasing model or feature complexity does not guarantee better performance  
- Effective solutions are driven by **data understanding, experimentation, and validation**  
- In this case, a simple unigram BoW representation was sufficient to achieve strong performance  